In [2]:
import torch
import hashlib
from PIL import Image
from IPython.display import display
from pathlib import Path
import unicodedata
import sys
import torch
import open_clip

import re
import os
import shutil
import numpy as np
import subprocess
from PIL import Image
from pocket_tts import TTSModel

from scipy.io import wavfile
import soundfile as sf
from datetime import timedelta

import sqlalchemy as sa
from sqlalchemy import text
from sqlalchemy.orm import Session

import boto3
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor

from PIL import Image as PILImage
import pillow_avif as vif 


In [3]:
root = Path.cwd()
while not (root / "src").exists():
    root = root.parent

sys.path.append(str(root))
from src.db.session import engine
from src.models import Pharaoh, PharaohScript  

In [4]:
root

WindowsPath('d:/GP/ECHO')

In [5]:
import gc
gc.collect()

3025

In [6]:
!nvidia-smi

Sat Mar 14 16:50:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.83                 Driver Version: 581.83         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   46C    P3             11W /   70W |       0MiB /   6141MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model, preprocess, tokenizer = open_clip.create_model_and_transforms(
    "ViT-H-14",
    pretrained="laion2b_s32b_b79k"
)
model = model.to(device)
model.eval()

CLIP(
  (visual): VisionTransformer(
    (conv1): Conv2d(3, 1280, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (patch_dropout): Identity()
    (ln_pre): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    (transformer): Transformer(
      (resblocks): ModuleList(
        (0-31): 32 x ResidualAttentionBlock(
          (ln_1): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=1280, out_features=1280, bias=True)
          )
          (ls_1): Identity()
          (ln_2): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
          (mlp): Sequential(
            (c_fc): Linear(in_features=1280, out_features=5120, bias=True)
            (gelu): GELU(approximate='none')
            (c_proj): Linear(in_features=5120, out_features=1280, bias=True)
          )
          (ls_2): Identity()
        )
      )
    )
    (ln_post): LayerNorm((1280,), eps=1e-05, elementwi

In [8]:
ENTITY_NAME = "Horus (God)"

with Session(engine) as session:
    script = (
        session.query(PharaohScript.pharaoh_script)
        .join(Pharaoh, Pharaoh.id == PharaohScript.pharaoh_id)
        .filter(sa.func.lower(Pharaoh.name) == ENTITY_NAME.lower())
        .limit(1)
        .scalar()
    )

if not script:
    raise ValueError(f"No script found for pharaoh name: {ENTITY_NAME}")

paragraphs = [p.strip() for p in re.split(r"\n\s*\n", script) if p.strip()]
paragraphs

["Horus, one of Egypt's oldest gods, has been revered for thousands of years. Early inscriptions show him with outstretched wings as protector to the nation’s rulers and in the serekh—a falcon on a perch—symbolizing his role from around 2850 B.C.E.",
 "This protective image was repeated throughout Egyptian history, famously seen in Khafre's statue where Horus stands guard. In later times, both Horus and Set were depicted as bringing the double crowns of Upper and Lower Egypt to pharaohs like Khafre.",
 "Horus’s cult centers thrived at temples such as Edfu, with its temple becoming a major site for worship from late predynastic times onward. The goddess Wadjet, protector of Lower Egypt, kept watch over Horus's divine family during their early years in the Nile delta.",
 'In battles and myths, Horus lost his left eye—the moon—during a fight with Set but was healed by Thoth, giving rise to the Wedjat (Eye of Horus) symbol still recognized today. This powerful myth explains why we see diff

In [9]:
#Text-to-Speech
tts_model = TTSModel.load_model()
voice_state = tts_model.get_state_for_audio_prompt("alba")

i = 0
for p in paragraphs:
    audio = tts_model.generate_audio(voice_state, p)
    wavfile.write(f"Outputs\\output_{i}.wav", tts_model.sample_rate, audio.numpy())
    i += 1

In [10]:
wav_files = sorted([f for f in os.listdir("Outputs") if f.endswith('.wav')])
print(wav_files)

audio_data = []
samplerate = None

for file_name in wav_files:
    file_name = os.path.join("Outputs", file_name)
    data, sr = sf.read(file_name)
    
    if samplerate is None:
        samplerate = sr
    elif sr != samplerate:
        raise ValueError("Sample rates do not match!")

    audio_data.append(data)

# Concatenate along time axis
combined = np.concatenate(audio_data, axis=0)

# Write output (keeps float format safely)
sf.write(f"Outputs\\{ENTITY_NAME}.wav", combined, samplerate)

['output_0.wav', 'output_1.wav', 'output_2.wav', 'output_3.wav', 'output_4.wav']


In [11]:
durations = []
images_needed = []
seconds = []
for i in range(len(paragraphs)):
    file_path = f"Outputs\\output_{i}.wav"
    # Returns the sample rate (Fs) and the data as a NumPy array
    Fs, data = wavfile.read(file_path) 
    # The length of the data array divided by the sample rate gives the duration
    duration_seconds = len(data) / float(Fs)
    durations.append(duration_seconds)
    images_needed.append(max(1, int(duration_seconds / 6)))
    section_seconds = duration_seconds / images_needed[-1]
    for img in range(images_needed[-1]):
        seconds.append(section_seconds)
    os.remove(file_path)

print(f"Durations of audio files: {durations}")
print(f"Images needed for each paragraph: {images_needed}")
print(f"Seconds for each image: {seconds}")

Durations of audio files: [16.4, 14.48, 15.36, 14.56, 5.84]
Images needed for each paragraph: [2, 2, 2, 2, 1]
Seconds for each image: [8.2, 8.2, 7.24, 7.24, 7.68, 7.68, 7.28, 7.28, 5.84]


In [12]:
i = 0
image_text_chunks = []
for p in paragraphs:
    sentences = re.split(r'(?<=[.!?]) +', p)
    sentences = [s.strip() for s in sentences if s.strip()]
    images_for_paragraph = images_needed[i]

    images_for_paragraph = min(images_for_paragraph, len(sentences))

    total_sentences = len(sentences)

    base = total_sentences // images_for_paragraph
    remainder = total_sentences % images_for_paragraph

    groups = []
    start = 0

    for img_idx in range(images_for_paragraph):
        extra = 1 if img_idx < remainder else 0
        end = start + base + extra

        chunk = " ".join(sentences[start:end])
        groups.append(chunk)

        start = end

    image_text_chunks.append(groups)
        
    i += 1
    print(f"Paragraph {i} split into {len(groups)} image text chunks.")
    print("-" * 40)
    print(groups)

Paragraph 1 split into 2 image text chunks.
----------------------------------------
["Horus, one of Egypt's oldest gods, has been revered for thousands of years.", 'Early inscriptions show him with outstretched wings as protector to the nation’s rulers and in the serekh—a falcon on a perch—symbolizing his role from around 2850 B.C.E.']
Paragraph 2 split into 2 image text chunks.
----------------------------------------
["This protective image was repeated throughout Egyptian history, famously seen in Khafre's statue where Horus stands guard.", 'In later times, both Horus and Set were depicted as bringing the double crowns of Upper and Lower Egypt to pharaohs like Khafre.']
Paragraph 3 split into 2 image text chunks.
----------------------------------------
['Horus’s cult centers thrived at temples such as Edfu, with its temple becoming a major site for worship from late predynastic times onward.', "The goddess Wadjet, protector of Lower Egypt, kept watch over Horus's divine family dur

In [13]:
name = ENTITY_NAME
fetched_images_ids = []
fetched_images_paths = []
for paragraph_chunks in image_text_chunks:
    for chunk in paragraph_chunks:
        text_tokens = open_clip.get_tokenizer()([chunk]).to(device)
    
        with torch.no_grad():
            text_embedding = model.encode_text(text_tokens)
            text_embedding /= text_embedding.norm(dim=-1, keepdim=True)

        embedding_list = text_embedding.cpu().numpy().tolist()[0]

        with Session(engine) as session:
            result = session.execute(text("SELECT pi.id, pi.image_path FROM pharaohs_images as pi JOIN pharaohs as p ON pi.pharaoh_id = p.id where p.name = :name ORDER BY image_embedding <=> CAST(:embedding AS vector)"), {"name": name, "embedding": embedding_list})
            pharaoh_images = result.fetchall()
            image_id = pharaoh_images[0][0]
            image_path = pharaoh_images[0][1]

            j = 0
            while image_id in fetched_images_ids:
                pharaoh_images.pop(0)
                if not pharaoh_images:
                    print("No more unique images available for this chunk.")
                    image_id = None
                    break
                image_id = pharaoh_images[0][0]
                image_path = pharaoh_images[0][1]
                j +=1
            fetched_images_ids.append(image_id)
            fetched_images_paths.append(image_path)
            
        print(f"Text chunk: {chunk}")
        print(f"Image ID: {image_id} on trial {j}")
        print(f"image path: {image_path}")

Text chunk: Horus, one of Egypt's oldest gods, has been revered for thousands of years.
Image ID: 305 on trial 0
image path: data/video_generation/raw/pharaohs_images/Horus (God)/Statue 1 angle 2.jpg
Text chunk: Early inscriptions show him with outstretched wings as protector to the nation’s rulers and in the serekh—a falcon on a perch—symbolizing his role from around 2850 B.C.E.
Image ID: 307 on trial 1
image path: data/video_generation/raw/pharaohs_images/Horus (God)/Statue 2.jpg
Text chunk: This protective image was repeated throughout Egyptian history, famously seen in Khafre's statue where Horus stands guard.
Image ID: 306 on trial 1
image path: data/video_generation/raw/pharaohs_images/Horus (God)/Statue 1.jpg
Text chunk: In later times, both Horus and Set were depicted as bringing the double crowns of Upper and Lower Egypt to pharaohs like Khafre.
Image ID: 297 on trial 3
image path: data/video_generation/raw/pharaohs_images/Horus (God)/Horus Relief 2.jpg
Text chunk: Horus’s cul

In [14]:
load_dotenv()

ACCOUNT_ID = os.getenv("R2_ACCOUNT_ID")
ACCESS_KEY = os.getenv("R2_ACCESS_KEY")
SECRET_KEY = os.getenv("R2_SECRET_KEY")
BUCKET_NAME = os.getenv("R2_BUCKET_NAME")

session = boto3.session.Session()
client = session.client(
    "s3",
    region_name="auto",
    endpoint_url=f"https://{ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=ACCESS_KEY,
    aws_secret_access_key=SECRET_KEY,
)

path = Path.joinpath(Path("data/video_generation/raw/pharaohs_images/"), name)
REMOTE_PREFIX = path

local_frames_dir = Path("temp_frames")
local_frames_dir.mkdir(exist_ok=True)

ordered_local_paths = []

def download_image(idx_key):
    idx, image_key = idx_key
    local_file = local_frames_dir / f"{idx:04d}.jpg"
    client.download_file(BUCKET_NAME, image_key, str(local_file))
    return str(local_file)

with ThreadPoolExecutor(max_workers=8) as executor:
    ordered_local_paths = list(
        executor.map(download_image, enumerate(fetched_images_paths))
    )

In [15]:
image_files = list(local_frames_dir.glob("*.jpg")) + list(local_frames_dir.glob("*.jpeg"))

In [16]:

for path in image_files:
    p = Path(path)
    try:
        with PILImage.open(p) as img:
            img = img.convert("RGB")  # handles AVIF, PNG, RGBA, etc.
            img.save(p, "JPEG", quality=95)
        print(f"Converted: {p}")
    except Exception as e:
        print(f"Failed to convert {p}: {e}")

Converted: temp_frames\0000.jpg
Converted: temp_frames\0001.jpg
Converted: temp_frames\0002.jpg
Converted: temp_frames\0003.jpg
Converted: temp_frames\0004.jpg
Converted: temp_frames\0005.jpg
Converted: temp_frames\0006.jpg
Converted: temp_frames\0007.jpg
Converted: temp_frames\0008.jpg


In [17]:
image_files

[WindowsPath('temp_frames/0000.jpg'),
 WindowsPath('temp_frames/0001.jpg'),
 WindowsPath('temp_frames/0002.jpg'),
 WindowsPath('temp_frames/0003.jpg'),
 WindowsPath('temp_frames/0004.jpg'),
 WindowsPath('temp_frames/0005.jpg'),
 WindowsPath('temp_frames/0006.jpg'),
 WindowsPath('temp_frames/0007.jpg'),
 WindowsPath('temp_frames/0008.jpg')]

In [18]:
# ---------- SETTINGS ----------
MAX_CHARS_PER_LINE = 42
MAX_LINES = 2
MIN_DURATION = 1.0  # minimum subtitle duration in seconds


# ---------- HELPERS ----------
def normalize_text(text):
    """
    Cleans problematic Unicode characters that break
    subtitle rendering in Pillow / MoviePy.
    """

    # First normalize Unicode composition
    text = unicodedata.normalize("NFKC", text)

    replacements = {
        "’": "'",
        "‘": "'",
        "‚": ",",
        "‛": "'",

        "“": '"',
        "”": '"',
        "„": '"',

        "—": "-",
        "–": "-",
        "―": "-",

        "…": "...",

        "\u00A0": " ",   # non-breaking space
        "\u200B": "",    # zero-width space
        "\u200C": "",
        "\u200D": "",
        "\uFEFF": "",    # BOM if embedded
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    # Remove any remaining control characters
    text = "".join(
        ch for ch in text
        if unicodedata.category(ch)[0] != "C"
    )

    return text

def format_timestamp(seconds):
    td = timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    millis = int((seconds - total_seconds) * 1000)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02}:{minutes:02}:{secs:02},{millis:03}"


def split_into_sentences(text):
    return re.split(r'(?<=[.!?]) +', text.strip())


def split_long_text(text, max_chars=MAX_CHARS_PER_LINE):
    words = text.split()
    lines = []
    current_line = ""

    for word in words:
        if len(current_line) + len(word) + 1 <= max_chars:
            current_line += (" " if current_line else "") + word
        else:
            lines.append(current_line)
            current_line = word

    if current_line:
        lines.append(current_line)

    # combine into blocks of max 2 lines
    blocks = []
    for i in range(0, len(lines), MAX_LINES):
        block = "\n".join(lines[i:i + MAX_LINES])
        blocks.append(block)

    return blocks


# ---------- MAIN FUNCTION ----------
def generate_srt(paragraphs, durations, output_path="subtitles.srt"):
    assert len(paragraphs) == len(durations), "Paragraphs and durations must match"

    current_time = 0.0
    subtitle_index = 1
    srt_blocks = []

    for paragraph, duration in zip(paragraphs, durations):
        paragraph = normalize_text(paragraph)
        sentences = split_into_sentences(paragraph)

        # break into subtitle chunks
        chunks = []
        for sentence in sentences:
            chunks.extend(split_long_text(sentence))

        total_chars = sum(len(chunk.replace("\n", "")) for chunk in chunks)

        for chunk in chunks:
            chunk_char_count = len(chunk.replace("\n", ""))

            # proportional timing
            chunk_duration = max(
                MIN_DURATION,
                (chunk_char_count / total_chars) * duration
            )

            start_time = current_time
            end_time = current_time + chunk_duration

            srt_block = f"""{subtitle_index}
{format_timestamp(start_time)} --> {format_timestamp(end_time)}
{chunk}

"""
            srt_blocks.append(srt_block)

            current_time = end_time
            subtitle_index += 1

    with open(output_path, "w", encoding="utf-8-sig") as f:
        f.writelines(srt_blocks)

    print(f"SRT file saved to {output_path}")


generate_srt(paragraphs, durations, "Outputs/output_subtitles.srt")

SRT file saved to Outputs/output_subtitles.srt


In [19]:
FFMPEG  = shutil.which("ffmpeg")

def run_ffmpeg(cmd):
    cmd = [FFMPEG if c == "ffmpeg" else str(c) for c in cmd]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)

'''
def run_ffmpeg(cmd):
    cmd = [FFMPEG if c == "ffmpeg" else str(c) for c in cmd]
    print("Running:", " ".join(cmd))

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    print("\n--- FFmpeg STDOUT ---\n")
    print(result.stdout)

    print("\n--- FFmpeg STDERR ---\n")
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"FFmpeg failed with exit code {result.returncode}")
'''

'\ndef run_ffmpeg(cmd):\n    cmd = [FFMPEG if c == "ffmpeg" else str(c) for c in cmd]\n    print("Running:", " ".join(cmd))\n\n    result = subprocess.run(\n        cmd,\n        stdout=subprocess.PIPE,\n        stderr=subprocess.PIPE,\n        text=True\n    )\n\n    print("\n--- FFmpeg STDOUT ---\n")\n    print(result.stdout)\n\n    print("\n--- FFmpeg STDERR ---\n")\n    print(result.stderr)\n\n    if result.returncode != 0:\n        raise RuntimeError(f"FFmpeg failed with exit code {result.returncode}")\n'

In [20]:
def _stable_int(seed: str) -> int:
    return int(hashlib.md5(seed.encode("utf-8")).hexdigest()[:8], 16)

def _stable_unit(seed: str) -> float:
    return (_stable_int(seed) % 10_000) / 10_000.0

In [21]:

def plan_kenburns_sequence(
    image_files,
    durations,
    target_w=1920,
    target_h=1080,
    threshold=40,
    min_pan_travel=140,
    max_pan_speed=120,
    vertical_travel_ratio=0.80,
    horizontal_travel_ratio=0.55,
):
    PALETTE = [
        ("zoom", "in"),
        ("pan",  "forward"),
        ("zoom", "out"),
        ("pan",  "reverse"),
    ]

    plans       = []
    palette_idx = 0
    last_sig    = None
    target_ar   = target_w / target_h

    for img_path, duration in zip(image_files, durations):
        img_path = Path(img_path)

        with Image.open(img_path) as img:
            img_w, img_h = img.size

        scale_factor = max(target_w / img_w, target_h / img_h)
        scaled_w     = int(round(img_w * scale_factor))
        scaled_h     = int(round(img_h * scale_factor))

        scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w + 1
        scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h + 1
        
        max_x        = max(0, scaled_w - target_w)
        max_y        = max(0, scaled_h - target_h)

        img_ar      = img_w / img_h
        is_vertical = img_ar < target_ar
        pan_axis    = "vertical" if is_vertical else "horizontal"

        travel_x = min(max_x, min(int(max_pan_speed * duration),
                                  int(max_x * horizontal_travel_ratio)))
        travel_y = min(max_y, min(int(max_pan_speed * duration),
                                  int(max_y * vertical_travel_ratio)))
        can_pan  = (travel_y > max(threshold, min_pan_travel)) if is_vertical \
                   else (travel_x > max(threshold, min_pan_travel))

        mode = direction = None
        for attempt in range(len(PALETTE)):
            cand_mode, cand_dir = PALETTE[(palette_idx + attempt) % len(PALETTE)]
            if cand_mode == "pan" and not can_pan:
                continue
            if (cand_mode, cand_dir) == last_sig:
                continue
            mode, direction = cand_mode, cand_dir
            palette_idx = (palette_idx + attempt + 1) % len(PALETTE)
            break

        if mode is None:
            mode        = "zoom"
            direction   = "out" if (last_sig and last_sig[1] == "in") else "in"
            palette_idx = (palette_idx + 1) % len(PALETTE)

        last_sig = (mode, direction)
        plans.append({
            "mode":        mode,
            "direction":   direction,
            "is_vertical": is_vertical,
            "pan_axis":    pan_axis,
        })

    return plans


In [22]:
def create_kenburns_clip(
    image_path,
    duration,
    output_path,
    fps=30,
    target_w=1920,
    target_h=1080,

    threshold=40,
    min_pan_travel=140,
    prefer_pan_when_possible=True,

    zoom_min=1.03,
    zoom_max=1.08,
    zoom_rate_per_sec=0.055,
    zoom_anchor_vertical_y=0.18,
    zoom_anchor_horizontal_y=0.50,

    max_pan_speed=200,
    vertical_travel_ratio=0.80,
    horizontal_travel_ratio=0.55,
    top_bias=0.00,

    motion_scale=1.4,
    use_nvenc=True,

    motion_mode=None,
    motion_direction=None,
):
    total_frames = max(2, int(round(duration * fps)))
    image_path   = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    with Image.open(image_path) as img:
        img_w, img_h = img.size

    scale_factor = max(target_w / img_w, target_h / img_h)
    scaled_w = (int(round(img_w * scale_factor)) )
    scaled_h = (int(round(img_h * scale_factor)) ) 

    scaled_w = scaled_w if scaled_w % 2 == 0 else scaled_w + 1
    scaled_h = scaled_h if scaled_h % 2 == 0 else scaled_h + 1
        
    max_x = max(0, scaled_w - target_w)
    max_y = max(0, scaled_h - target_h)

    denom    = total_frames - 1
    ease_pan  = f"(0.5-0.5*cos(PI*n/{denom}))"
    ease_zoom = f"(0.5-0.5*cos(PI*on/{denom}))"

    target_ar   = target_w / target_h
    img_ar      = img_w / img_h
    is_vertical = img_ar < target_ar

    u = _stable_unit(str(image_path))

    zoom_direction = "out" if u < 0.5 else "in"
    pan_reverse    = u > 0.66

    if motion_direction is not None:
        if motion_direction in ("in", "out"):
            zoom_direction = motion_direction
        elif motion_direction in ("forward", "reverse"):
            pan_reverse = (motion_direction == "reverse")

    dyn_cap = 1.0 + zoom_rate_per_sec * duration
    zmax    = min(zoom_max, dyn_cap)
    z_target = max(zoom_min, min(zoom_min + (zmax - zoom_min) * (0.25 + 0.50 * u), zmax))

    travel_x = min(max_x, min(int(max_pan_speed * duration), int(max_x * horizontal_travel_ratio)))
    travel_y = min(max_y, min(int(max_pan_speed * duration), int(max_y * vertical_travel_ratio)))
    can_pan  = (travel_y > max(threshold, min_pan_travel)) if is_vertical \
               else (travel_x > max(threshold, min_pan_travel))

    ms                     = max(1.0, float(motion_scale))
    mw, mh                 = int(round(target_w * ms)) , int(round(target_h * ms))
    scaled_w2, scaled_h2   = int(round(scaled_w * ms)), int(round(scaled_h * ms))
    max_x2, max_y2         = max(0, scaled_w2 - mw), max(0, scaled_h2 - mh)
    travel_x2, travel_y2   = int(round(min(max_x2, travel_x * ms))),  int(round(min(max_y2, travel_y * ms)))

    def build_pan_vf():
        if is_vertical  and travel_y2 < 220: return None
        if not is_vertical and travel_x2 < 220: return None

        if is_vertical:
            start_y = int(round(max_y2 * top_bias))
            end_y   = min(start_y + travel_y2, max_y2)
            if pan_reverse: start_y, end_y = end_y, start_y
            x_expr = f"{max_x2 // 2}"
            y_expr = f"trunc({start_y}+({end_y}-{start_y})*{ease_pan})"
        else:
            start_x, end_x = 0, travel_x2
            if pan_reverse: start_x, end_x = end_x, start_x
            x_expr = f"trunc({start_x}+({end_x}-{start_x})*{ease_pan})"
            y_expr = f"{max_y2 // 2}"

        return (
            f"scale={scaled_w2}:{scaled_h2}:flags=lanczos,"
            f"crop={mw}:{mh}:x='{x_expr}':y='{y_expr}',"
            f"scale={target_w}:{target_h}:flags=lanczos,"
            f"setsar=1,setdar=16/9,format=yuv420p"
        )

    def build_zoom_vf():
        if zoom_direction == "out":
            z_expr = f"{z_target}-({z_target}-1.0)*{ease_zoom}"
        else:
            z_expr = f"1.0+({z_target}-1.0)*{ease_zoom}"

        ax = 0.5
        ay = zoom_anchor_vertical_y if is_vertical else zoom_anchor_horizontal_y
        x0 = f"max(0,min(({ax})*iw-ow/2,iw-ow))"
        y0 = f"max(0,min(({ay})*ih-oh/2,ih-oh))"

        return (
            f"scale={scaled_w2}:{scaled_h2}:flags=lanczos,"
            f"zoompan=z='{z_expr}':x='if(eq(on,1),{x0},px)':y='if(eq(on,1),{y0},py)'"
            f":d=1:s={mw}x{mh}:fps={fps},"
            f"scale={target_w}:{target_h}:flags=lanczos,"
            f"setsar=1,setdar=16/9,format=yuv420p"
        )

    if motion_mode == "pan":
        vf = build_pan_vf() or build_zoom_vf()
    elif motion_mode == "zoom":
        vf = build_zoom_vf()
    else:
        vf = (build_pan_vf() or build_zoom_vf()) if (prefer_pan_when_possible and can_pan) \
             else build_zoom_vf()

    vcodec = ["-c:v", "h264_nvenc", "-preset", "p1"] if use_nvenc \
             else ["-c:v", "libx264", "-preset", "slow", "-crf", "18"]

    run_ffmpeg([
        "ffmpeg", "-y",
        "-loop", "1", "-framerate", str(fps), "-t", str(duration), "-i", str(image_path),
        "-vf", vf, "-frames:v", str(total_frames),
        *vcodec, "-pix_fmt", "yuv420p","-fps_mode", "cfr",
        output_path,
    ])



In [23]:
'''def generate_all_clips(image_files, durations, temp_dir="temp_clips"):
    temp_dir = Path(temp_dir)
    temp_dir.mkdir(exist_ok=True)

    outputs = []
    for i, (img, dur) in enumerate(zip(image_files, durations)):
        out = temp_dir / f"clip_{i}.mp4"
        create_kenburns_clip(img, dur, out)
        outputs.append(str(out))

    return outputs

def concatenate_clips(clips, output_path):
    """
    Safe concat with hard cuts.
    This avoids all xfade timeline errors.
    """
    list_file = Path("Outputs") / "concat_list.txt"

    with open(list_file, "w", encoding="utf-8") as f:
        for clip in clips:
            f.write(f"file '{os.path.abspath(clip)}'\n")

    cmd = [
        "ffmpeg",
        "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", str(list_file),
        "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-pix_fmt", "yuv420p",
        output_path
    ]

    run_ffmpeg(cmd)'''

'def generate_all_clips(image_files, durations, temp_dir="temp_clips"):\n    temp_dir = Path(temp_dir)\n    temp_dir.mkdir(exist_ok=True)\n\n    outputs = []\n    for i, (img, dur) in enumerate(zip(image_files, durations)):\n        out = temp_dir / f"clip_{i}.mp4"\n        create_kenburns_clip(img, dur, out)\n        outputs.append(str(out))\n\n    return outputs\n\ndef concatenate_clips(clips, output_path):\n    """\n    Safe concat with hard cuts.\n    This avoids all xfade timeline errors.\n    """\n    list_file = Path("Outputs") / "concat_list.txt"\n\n    with open(list_file, "w", encoding="utf-8") as f:\n        for clip in clips:\n            f.write(f"file \'{os.path.abspath(clip)}\'\n")\n\n    cmd = [\n        "ffmpeg",\n        "-y",\n        "-f", "concat",\n        "-safe", "0",\n        "-i", str(list_file),\n        "-c:v", "h264_nvenc",\n        "-preset", "p1",\n        "-pix_fmt", "yuv420p",\n        output_path\n    ]\n\n    run_ffmpeg(cmd)'

In [24]:
def generate_all_clips(image_files, durations, temp_dir="temp_clips"):
    Path(temp_dir).mkdir(exist_ok=True)

    plans = plan_kenburns_sequence(image_files, durations)

    outputs = []
    for i, (img, dur, plan) in enumerate(zip(image_files, durations, plans)):
        out = f"{temp_dir}/clip_{i}.mp4"

        create_kenburns_clip(
            image_path=img,
            duration=dur,
            output_path=out,
            motion_mode=plan["mode"],
            motion_direction=plan["direction"],
        )
        outputs.append(out)

    return outputs

'''def concatenate_clips(clips, output_path):
    list_file = "Outputs/concat_list.txt"

    with open(list_file, "w") as f:
        for clip in clips:
            f.write(f"file '{os.path.abspath(clip)}'\n")

    cmd = [
        "ffmpeg",
        "-y",
        "-f", "concat",
        "-safe", "0",
        "-i", list_file,
        "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)'''

'def concatenate_clips(clips, output_path):\n    list_file = "Outputs/concat_list.txt"\n\n    with open(list_file, "w") as f:\n        for clip in clips:\n            f.write(f"file \'{os.path.abspath(clip)}\'\n")\n\n    cmd = [\n        "ffmpeg",\n        "-y",\n        "-f", "concat",\n        "-safe", "0",\n        "-i", list_file,\n        "-c:v", "h264_nvenc",\n        "-preset", "p1",\n        "-c:a", "aac",\n        output_path\n    ]\n\n    run_ffmpeg(cmd)'

In [25]:
def distribute_durations_exact(audio_duration, n_clips, fade):
    target_total = audio_duration + fade * (n_clips - 1)
    base = target_total / n_clips
    durations = [base] * n_clips
    durations[-1] += target_total - sum(durations)  # fix float rounding
    return durations

In [26]:
def get_duration(path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    return float(result.stdout.strip())


In [27]:
def concatenate_clips(clips, output_path, fade=0.45):

    durations_local = [get_duration(c) for c in clips]

    cmd = ["ffmpeg", "-y"]
    for c in clips:
        cmd += ["-i", str(c)]

    filter_parts = []

    # IMPORTANT FIX: normalize every clip before xfade
    filter_parts.append(
        "[0:v]fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p[v0]"
    )

    for i in range(1, len(clips)):
        filter_parts.append(
            f"[{i}:v]fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p[v{i}src]"
        )

    cumulative = durations_local[0]
    last = "v0"

    for i in range(1, len(clips)):
        offset = cumulative - fade
        out = f"vx{i}"
        filter_parts.append(
            f"[{last}][v{i}src]xfade=transition=fade:duration={fade}:offset={offset}[{out}]"
        )
        cumulative += durations_local[i] - fade
        last = out

    filter_complex = ";".join(filter_parts)

    cmd += [
        "-safe", "0",
        "-filter_complex", filter_complex,
        "-map", f"[{last}]",
        "-c:v", "h264_nvenc",
        "-preset", "medium",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
      
        str(output_path)
    ]

    run_ffmpeg(cmd)

In [28]:
def add_audio(video_path, audio_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-i", audio_path,
        "-c:v", "copy",
        "-c:a", "aac",
        output_path
    ]

    run_ffmpeg(cmd)

def add_subtitles(video_path, srt_path, output_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-vf", f"subtitles={srt_path}",
        "-c:v", "h264_nvenc",
        "-preset", "p1",
        "-c:a", "copy",
        output_path
    ]

    run_ffmpeg(cmd)

def cleanup_files():
    temp_dir = "temp_clips"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    temp_dir = "temp_frames"
    if os.path.exists(temp_dir):
        for f in os.listdir(temp_dir):
            os.remove(os.path.join(temp_dir, f))
        os.rmdir(temp_dir)

    for f in os.listdir('Outputs'):
        if f.startswith("combined") or f.startswith("with_audio") or f.startswith("concat_list") or f.startswith("output_subtitles") or f.endswith(".wav"):
            os.remove(os.path.join('Outputs', f))

In [29]:
TEMP_DIR = Path("temp_frames")
TEMP_CLIP = Path("temp_clips")
OUT_DIR = Path("Outputs")

In [30]:
print("Working dir:", Path.cwd())

print("Temp images exist?", TEMP_DIR.exists() and any(TEMP_DIR.iterdir()))
print("Temp clips exist?", TEMP_CLIP.exists() and any(TEMP_CLIP.iterdir()))
print("combined.mp4?", (OUT_DIR / "combined.mp4").exists())
print("audio wav?", (OUT_DIR / f"{ENTITY_NAME}.wav").exists())
print("subtitles srt?", (OUT_DIR / "output_subtitles.srt").exists())
print("Final concatenated output ?", (OUT_DIR / "final_video.mp4").exists())

print("len(image_files) =", len(image_files))
print("len(seconds) =", len(seconds))

Working dir: d:\GP\ECHO\experiments\video_generation\video_generation_pharaohs
Temp images exist? True
Temp clips exist? False
combined.mp4? False
audio wav? True
subtitles srt? True
Final concatenated output ? True
len(image_files) = 9
len(seconds) = 9


In [31]:
audio_file = f"Outputs/{ENTITY_NAME}.wav"
audio_duration = get_duration(audio_file)

seconds = distribute_durations_exact(
    audio_duration=audio_duration,
    n_clips=len(image_files),
    fade=0.45
)

# 1. Generate Ken Burns clips
clips = generate_all_clips(image_files, seconds)

# 2. Concatenate
concatenated = "Outputs/combined.mp4"
concatenate_clips(clips, concatenated)

# 3. Add audio
with_audio = "Outputs/with_audio.mp4"
add_audio(concatenated, f"Outputs/{ENTITY_NAME}.wav", with_audio)

# 4. Add subtitles
final_output = "Outputs/final_video.mp4"
add_subtitles(with_audio, "Outputs/output_subtitles.srt", final_output)

print("Done:", final_output)

Running: C:\Users\Malak Diab\Downloads\ffmpeg-2026-03-01-git-862338fe31-full_build\ffmpeg-2026-03-01-git-862338fe31-full_build\bin\ffmpeg.EXE -y -loop 1 -framerate 30 -t 7.804444444444444 -i temp_frames\0000.jpg -vf scale=2688:4032:flags=lanczos,zoompan=z='1.0+(1.0475775-1.0)*(0.5-0.5*cos(PI*on/233))':x='if(eq(on,1),max(0,min((0.5)*iw-ow/2,iw-ow)),px)':y='if(eq(on,1),max(0,min((0.18)*ih-oh/2,ih-oh)),py)':d=1:s=2688x1512:fps=30,scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p -frames:v 234 -c:v h264_nvenc -preset p1 -pix_fmt yuv420p -fps_mode cfr temp_clips/clip_0.mp4
Running: C:\Users\Malak Diab\Downloads\ffmpeg-2026-03-01-git-862338fe31-full_build\ffmpeg-2026-03-01-git-862338fe31-full_build\bin\ffmpeg.EXE -y -loop 1 -framerate 30 -t 7.804444444444444 -i temp_frames\0001.jpg -vf scale=2688:4049:flags=lanczos,crop=2688:1512:x='0':y='trunc(0+(2029-0)*(0.5-0.5*cos(PI*n/233)))',scale=1920:1080:flags=lanczos,setsar=1,setdar=16/9,format=yuv420p -frames:v 234 -c:v h264_nvenc 

In [32]:
print("Working dir:", Path.cwd())

print("Temp images exist?", TEMP_DIR.exists() and any(TEMP_DIR.iterdir()))
print("Temp clips exist?", TEMP_DIR.exists() and any(TEMP_DIR.iterdir()))
print("combined.mp4?", (OUT_DIR / "combined.mp4").exists())
print("audio wav?", (OUT_DIR / f"{ENTITY_NAME}.wav").exists())
print("subtitles srt?", (OUT_DIR / "output_subtitles.srt").exists())
print("Final concatenated output ?", (OUT_DIR / "final_video.mp4").exists())

print("len(image_files) =", len(image_files))
print("len(seconds) =", len(seconds))

Working dir: d:\GP\ECHO\experiments\video_generation\video_generation_pharaohs
Temp images exist? True
Temp clips exist? True
combined.mp4? True
audio wav? True
subtitles srt? True
Final concatenated output ? True
len(image_files) = 9
len(seconds) = 9


In [33]:
#cleanup_files()